In [3]:
# ============================================================
# 18_compare_tfg_vs_respicast_corrected.ipynb
#
# comparison between TFG final-test forecasts and RespiCast participant forecasts.
#
#
# 2) RespiCast origin dates are publication Wednesdays.
#    The comparable TFG origin is the previous Sunday, not the next Sunday.
#
# 3) RespiCast horizons are mapped through dates:
#       origin_tfg = previous Sunday before/at origin_raw
#       target     = target_end_date normalized to Sunday
#       h_tfg      = (target - origin_tfg) / 7
#
#    Therefore RespiCast horizon 1 usually maps to TFG h=0
#    and is excluded from the main h=1..4 comparison.
#    RespiCast horizons 2,3,4 usually map to TFG h=1,2,3.
#
# 4) Main ranking is based on exact paired rows against
#    respicast-quantileBaseline.
#
# Main score:
#       log2(AE_ref / AE_model)
#
# Positive => model improves over RespiCast quantile baseline.
# Negative => model is worse than RespiCast quantile baseline.
# ============================================================

import sys
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 0. Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src.config import get_project_paths
    paths = get_project_paths(PROJECT_ROOT)
    RESULTS_DIR = paths.results
except Exception:
    RESULTS_DIR = PROJECT_ROOT / "results"

FINAL_TEST_DIR = RESULTS_DIR / "final_test_2024_2025"

RESPICAST_REPO_CANDIDATES = [
    PROJECT_ROOT / "RespiCast-SyndromicIndicators-main",
    PROJECT_ROOT / "RespiCast-SyndromicIndicators",
    PROJECT_ROOT / "respicast-syndromicindicators-main",
]

RESPICAST_REPO = None
for p in RESPICAST_REPO_CANDIDATES:
    if p.exists() and (p / "model-output").exists():
        RESPICAST_REPO = p
        break

if RESPICAST_REPO is None:
    found = [p for p in PROJECT_ROOT.glob("RespiCast*") if (p / "model-output").exists()]
    if len(found) > 0:
        RESPICAST_REPO = found[0]

if RESPICAST_REPO is None:
    raise FileNotFoundError(
        "Could not find RespiCast repo. Expected folder like "
        "'RespiCast-SyndromicIndicators-main' at project root."
    )

MODEL_OUTPUT_DIR = RESPICAST_REPO / "model-output"

OUT_DIR = FINAL_TEST_DIR / "respicast_comparison_corrected"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TFG_PRED_PATH = FINAL_TEST_DIR / "finaltest_all_predictions_long.csv"
MASE_SCALE_PATH = FINAL_TEST_DIR / "finaltest_mase_scales_2014_2023.csv"

if not TFG_PRED_PATH.exists():
    raise FileNotFoundError(TFG_PRED_PATH)

if not MASE_SCALE_PATH.exists():
    raise FileNotFoundError(MASE_SCALE_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_TEST_DIR:", FINAL_TEST_DIR)
print("RESPICAST_REPO:", RESPICAST_REPO)
print("MODEL_OUTPUT_DIR:", MODEL_OUTPUT_DIR)
print("OUT_DIR:", OUT_DIR)


# ============================================================
# 1. Configuration
# ============================================================

TFG_HORIZONS = [1, 2, 3, 4]
COMMON6 = ["BE", "CZ", "EE", "LT", "RO", "SI"]

REFERENCE_MODEL_CONTAINS = "quantilebaseline"

EPS_AE = 1e-8

# For main filtered ranking. This is coverage over the reference baseline rows,
# not over the complete TFG test period.
MIN_COVERAGE_VS_REFERENCE = 0.50

# Extreme prediction audit threshold. Forecasts are not removed just because
# they are extreme; they are only reported.
EXTREME_PRED_ABS_THRESHOLD = 1e6


# ============================================================
# 2. Date helpers
# ============================================================

def previous_sunday(dates):
    """
    For publication/origin dates:
    map each date to the previous Sunday, including same date if already Sunday.
    
    Example:
        2024-10-23 Wednesday -> 2024-10-20 Sunday
    """
    d = pd.to_datetime(dates, errors="coerce")
    days_since_sunday = (d.dt.weekday - 6) % 7
    return d - pd.to_timedelta(days_since_sunday, unit="D")


def nearest_sunday(dates):
    """
    For target_end_date:
    normalize to nearest Sunday. In the RespiCast files inspected,
    target_end_date is already Sunday. This function is defensive.
    """
    d = pd.to_datetime(dates, errors="coerce")
    weekday = d.dt.weekday

    prev_days = (weekday - 6) % 7
    next_days = (6 - weekday) % 7

    use_next = next_days < prev_days

    out = d - pd.to_timedelta(prev_days, unit="D")
    out.loc[use_next] = d.loc[use_next] + pd.to_timedelta(next_days.loc[use_next], unit="D")
    return out


def find_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


def parse_file_date(path):
    m = re.search(r"(\d{4}-\d{2}-\d{2})", path.name)
    if m:
        return pd.to_datetime(m.group(1))
    return pd.NaT


def infer_disease_from_target(target_value):
    if pd.isna(target_value):
        return None

    s = str(target_value).lower()

    if re.search(r"\bili\b", s) or "influenza-like" in s or "influenza like" in s:
        return "ILI"

    if re.search(r"\bari\b", s) or "acute respiratory" in s:
        return "ARI"

    return None


def parse_numeric_horizon(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x)
    m = re.search(r"-?\d+", s)
    if m:
        return float(m.group(0))
    return np.nan


def safe_log2_ratio(ae_ref, ae_model, eps=EPS_AE):
    """
    Computes log2(AE_ref / AE_model) robustly.

    If both errors are exactly zero, score is set to 0.
    Otherwise, eps is used only to avoid undefined divisions/logs.
    """
    ae_ref = np.asarray(ae_ref, dtype=float)
    ae_model = np.asarray(ae_model, dtype=float)

    both_zero = (ae_ref <= 0) & (ae_model <= 0)

    ref_safe = np.maximum(ae_ref, eps)
    model_safe = np.maximum(ae_model, eps)

    out = np.log2(ref_safe / model_safe)
    out[both_zero] = 0.0
    return out


# ============================================================
# 3. Load and clean TFG final predictions
# ============================================================

tfg_raw = pd.read_csv(TFG_PRED_PATH)
tfg_raw.columns = [c.strip() for c in tfg_raw.columns]

required_tfg_cols = ["origin", "target", "disease", "location", "h", "y", "pred", "model"]
missing = [c for c in required_tfg_cols if c not in tfg_raw.columns]
if missing:
    raise ValueError(f"Missing required TFG columns: {missing}")

tfg = tfg_raw.copy()

tfg["origin"] = pd.to_datetime(tfg["origin"], errors="coerce")
tfg["target"] = pd.to_datetime(tfg["target"], errors="coerce")
tfg["disease"] = tfg["disease"].astype(str).str.upper()
tfg["location"] = tfg["location"].astype(str)
tfg["model"] = tfg["model"].astype(str)
tfg["h"] = pd.to_numeric(tfg["h"], errors="coerce")
tfg["y"] = pd.to_numeric(tfg["y"], errors="coerce")
tfg["pred"] = pd.to_numeric(tfg["pred"], errors="coerce")

rows_before_clean = len(tfg)

# Critical correction:
# Remove predictions for targets where no observed y exists,
# and remove non-finite predictions.
tfg = tfg.dropna(subset=["origin", "target", "disease", "location", "h", "y", "pred"]).copy()
tfg = tfg[np.isfinite(tfg["y"]) & np.isfinite(tfg["pred"])].copy()
tfg["h"] = tfg["h"].astype(int)
tfg = tfg[tfg["h"].isin(TFG_HORIZONS)].copy()

rows_after_clean = len(tfg)

print("\n============================================================")
print("TFG prediction cleaning")
print("============================================================")
print("Original rows:", rows_before_clean)
print("Rows after dropping missing y/pred and non-finite values:", rows_after_clean)
print("Rows removed:", rows_before_clean - rows_after_clean)

# Verify no duplicates after cleaning
tfg_key = ["origin", "target", "disease", "location", "h", "model"]

dup = (
    tfg.groupby(tfg_key, as_index=False)
       .agg(
           n_rows=("pred", "size"),
           n_unique_pred=("pred", "nunique"),
           n_unique_y=("y", "nunique")
       )
)
dup = dup[dup["n_rows"] > 1].copy()

if len(dup) > 0:
    print("\nDuplicated TFG prediction keys found after cleaning:")
    display(dup.head(30))
    raise ValueError("TFG predictions still contain duplicate model-task rows.")

# Verify horizon-date consistency
tfg["date_diff_weeks"] = ((tfg["target"] - tfg["origin"]).dt.days / 7.0)
bad_tfg_h = tfg[~np.isclose(tfg["date_diff_weeks"], tfg["h"])].copy()

if len(bad_tfg_h) > 0:
    print("\nBad TFG horizon/date rows:")
    display(bad_tfg_h.head(30))
    raise ValueError("Some TFG rows have inconsistent h and target-origin date difference.")

# Truth universe from cleaned TFG predictions
truth_keys = ["origin", "target", "disease", "location", "h"]

truth_universe = (
    tfg[truth_keys + ["y"]]
    .drop_duplicates(truth_keys)
    .reset_index(drop=True)
)

truth_y_check = (
    tfg[truth_keys + ["y"]]
    .groupby(truth_keys, as_index=False)
    .agg(n_unique_y=("y", "nunique"))
)
bad_y = truth_y_check[truth_y_check["n_unique_y"] > 1].copy()

if len(bad_y) > 0:
    display(bad_y.head(30))
    raise ValueError("Same truth key has more than one y value.")

# Coverage audit after cleaning
tfg_coverage = (
    tfg.groupby(["model", "disease", "h"], as_index=False)
       .agg(n_points=("pred", "size"), n_countries=("location", "nunique"))
)

truth_counts = (
    truth_universe.groupby(["disease", "h"], as_index=False)
    .agg(tfg_universe_points=("y", "size"), tfg_universe_countries=("location", "nunique"))
)

tfg_coverage = tfg_coverage.merge(truth_counts, on=["disease", "h"], how="left")
tfg_coverage["coverage_vs_tfg_universe"] = tfg_coverage["n_points"] / tfg_coverage["tfg_universe_points"]

print("\nTFG coverage after cleaning:")
display(tfg_coverage.sort_values(["disease", "h", "model"]))

if (tfg_coverage["coverage_vs_tfg_universe"] > 1.0000001).any():
    display(tfg_coverage[tfg_coverage["coverage_vs_tfg_universe"] > 1.0000001])
    raise ValueError("Coverage > 1 still exists after TFG cleaning.")

print("\nTFG truth universe after cleaning:")
display(
    truth_universe.groupby(["disease", "h"], as_index=False)
    .agg(
        n_points=("y", "size"),
        n_countries=("location", "nunique"),
        first_origin=("origin", "min"),
        last_origin=("origin", "max"),
        first_target=("target", "min"),
        last_target=("target", "max")
    )
)

tfg_predictions = tfg[truth_keys + ["y", "pred", "model"]].copy()
tfg_predictions["source"] = "TFG"
tfg_predictions["method"] = "TFG::" + tfg_predictions["model"].astype(str)


# ============================================================
# 4. Load MASE scales
# ============================================================

mase_scales = pd.read_csv(MASE_SCALE_PATH)
mase_scales.columns = [c.strip() for c in mase_scales.columns]

required_scale_cols = ["disease", "location", "mase_scale"]
missing_scales = [c for c in required_scale_cols if c not in mase_scales.columns]
if missing_scales:
    raise ValueError(f"Missing MASE scale columns: {missing_scales}")

mase_scales["disease"] = mase_scales["disease"].astype(str).str.upper()
mase_scales["location"] = mase_scales["location"].astype(str)
mase_scales["mase_scale"] = pd.to_numeric(mase_scales["mase_scale"], errors="coerce")

mase_scales = mase_scales.dropna(subset=["mase_scale"]).copy()

print("\nMASE scales loaded:", mase_scales.shape)


# ============================================================
# 5. Read RespiCast forecasts with corrected origin mapping
# ============================================================

def read_respicast_file_corrected(path):
    """
    Reads one RespiCast model-output CSV and returns point-like forecasts:
    - median
    - point / mean
    - quantile 0.5

    Corrected time mapping:
    - origin_raw = publication date, usually Wednesday
    - origin     = previous Sunday, comparable to TFG origin
    - target     = target_end_date normalized to Sunday
    - h          = (target - origin) / 7
    """
    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as e:
        return None, {
            "file": str(path),
            "model": path.parent.name,
            "status": "read_error",
            "error": str(e)[:300],
            "raw_rows": 0,
            "usable_rows": 0
        }

    raw_rows = len(df)

    if raw_rows == 0:
        return None, {
            "file": str(path),
            "model": path.parent.name,
            "status": "empty",
            "error": "",
            "raw_rows": 0,
            "usable_rows": 0
        }

    df.columns = [c.strip() for c in df.columns]

    origin_col = find_col(df, ["origin_date", "forecast_date", "reference_date"])
    target_end_col = find_col(df, ["target_end_date", "target_date", "date"])
    target_col = find_col(df, ["target"])
    horizon_col = find_col(df, ["horizon"])
    location_col = find_col(df, ["location"])
    output_type_col = find_col(df, ["output_type"])
    output_type_id_col = find_col(df, ["output_type_id"])
    value_col = find_col(df, ["value", "prediction", "pred"])

    required = {
        "origin_date": origin_col,
        "target_end_date": target_end_col,
        "target": target_col,
        "location": location_col,
        "output_type": output_type_col,
        "value": value_col,
    }

    missing = [k for k, v in required.items() if v is None]

    if missing:
        return None, {
            "file": str(path),
            "model": path.parent.name,
            "status": "missing_required_columns",
            "error": f"Missing {missing}; columns={list(df.columns)}",
            "raw_rows": raw_rows,
            "usable_rows": 0
        }

    out = pd.DataFrame()
    out["origin_raw"] = pd.to_datetime(df[origin_col], errors="coerce")
    out["target_raw"] = pd.to_datetime(df[target_end_col], errors="coerce")

    # Corrected origin mapping
    out["origin"] = previous_sunday(out["origin_raw"]) - pd.Timedelta(days=7)

    # Target is expected to already be Sunday; normalize defensively.
    out["target"] = nearest_sunday(out["target_raw"])
    out["target_shift_days"] = (out["target"] - out["target_raw"]).dt.days

    out["target_name"] = df[target_col].astype(str)
    out["disease"] = df[target_col].apply(infer_disease_from_target)
    out["location"] = df[location_col].astype(str)

    out["pred"] = pd.to_numeric(df[value_col], errors="coerce")

    out["output_type"] = df[output_type_col].astype(str).str.lower().str.strip()

    if output_type_id_col is not None:
        out["output_type_id"] = df[output_type_id_col].astype(str)
        out["output_type_id_num"] = pd.to_numeric(df[output_type_id_col], errors="coerce")
    else:
        out["output_type_id"] = ""
        out["output_type_id_num"] = np.nan

    if horizon_col is not None:
        out["respicast_horizon_raw"] = df[horizon_col].astype(str)
        out["respicast_horizon_num"] = df[horizon_col].apply(parse_numeric_horizon)
    else:
        out["respicast_horizon_raw"] = ""
        out["respicast_horizon_num"] = np.nan

    # Corrected TFG horizon
    out["date_diff_weeks_from_corrected_origin"] = ((out["target"] - out["origin"]).dt.days / 7.0)

    out["h_is_integer"] = np.isclose(
        out["date_diff_weeks_from_corrected_origin"],
        np.round(out["date_diff_weeks_from_corrected_origin"]),
        atol=1e-8
    )

    out["h"] = np.round(out["date_diff_weeks_from_corrected_origin"]).astype("float")

    # Select point-like forecasts
    is_median = out["output_type"].eq("median")
    is_point_or_mean = out["output_type"].isin(["point", "mean"])
    is_quantile_05 = (
        out["output_type"].eq("quantile")
        & np.isclose(out["output_type_id_num"], 0.5, atol=1e-10)
    )

    keep_point = is_median | is_point_or_mean | is_quantile_05

    out = out[keep_point].copy()

    out["median_source_type"] = np.select(
        [
            out["output_type"].eq("median"),
            out["output_type"].isin(["point", "mean"]),
            out["output_type"].eq("quantile") & np.isclose(out["output_type_id_num"], 0.5, atol=1e-10),
        ],
        ["median", "point_or_mean", "quantile_0.5"],
        default="other"
    )

    out["median_priority"] = np.select(
        [
            out["median_source_type"].eq("median"),
            out["median_source_type"].eq("point_or_mean"),
            out["median_source_type"].eq("quantile_0.5"),
        ],
        [0, 1, 2],
        default=9
    )

    out["model"] = path.parent.name
    out["source"] = "RespiCast"
    out["method"] = "RespiCast::" + out["model"].astype(str)
    out["file"] = str(path)
    out["file_name"] = path.name
    out["file_date"] = parse_file_date(path)

    out = out.dropna(subset=["origin", "target", "disease", "location", "h", "pred"]).copy()
    out = out[np.isfinite(out["pred"])].copy()
    out["h"] = out["h"].astype(int)

    # Main comparison uses the same h=1..4 convention as TFG.
    # h<=0 are nowcast/backcast from RespiCast relative to previous Sunday.
    out = out[out["h"].isin(TFG_HORIZONS)].copy()
    out = out[out["disease"].isin(["ARI", "ILI"])].copy()
    out = out[out["h_is_integer"]].copy()

    # Deduplicate within same model-task. Prefer median > point/mean > quantile 0.5.
    out = (
        out.sort_values(
            [
                "model", "disease", "location", "origin", "target", "h",
                "median_priority", "file_date"
            ],
            ascending=[True, True, True, True, True, True, True, False]
        )
        .drop_duplicates(["model", "disease", "location", "origin", "target", "h"], keep="first")
        .reset_index(drop=True)
    )

    status = {
        "file": str(path),
        "model": path.parent.name,
        "status": "ok",
        "error": "",
        "raw_rows": raw_rows,
        "usable_rows": len(out),
        "n_ari_rows": int((out["disease"] == "ARI").sum()) if len(out) else 0,
        "n_ili_rows": int((out["disease"] == "ILI").sum()) if len(out) else 0,
        "first_origin_raw": out["origin_raw"].min() if len(out) else pd.NaT,
        "last_origin_raw": out["origin_raw"].max() if len(out) else pd.NaT,
        "first_origin_corrected": out["origin"].min() if len(out) else pd.NaT,
        "last_origin_corrected": out["origin"].max() if len(out) else pd.NaT,
        "first_target": out["target"].min() if len(out) else pd.NaT,
        "last_target": out["target"].max() if len(out) else pd.NaT,
        "h_values": ",".join(map(str, sorted(out["h"].unique()))) if len(out) else "",
        "max_abs_target_shift_days": int(out["target_shift_days"].abs().max()) if len(out) else 0,
    }

    return out, status


all_respicast_csvs = sorted(MODEL_OUTPUT_DIR.rglob("*.csv"))

print("\n============================================================")
print("Loading RespiCast forecasts")
print("============================================================")
print("CSV files found:", len(all_respicast_csvs))

respicast_parts = []
inventory_rows = []

for i, path in enumerate(all_respicast_csvs, start=1):
    if i % 200 == 0:
        print(f"Processed {i}/{len(all_respicast_csvs)} files...")

    part, status = read_respicast_file_corrected(path)
    inventory_rows.append(status)

    if part is not None and len(part) > 0:
        respicast_parts.append(part)

respicast_inventory = pd.DataFrame(inventory_rows)

if len(respicast_parts) == 0:
    raise ValueError("No usable RespiCast predictions loaded after corrected mapping.")

respicast_pred = pd.concat(respicast_parts, ignore_index=True)

# Global deduplication across files
respicast_pred = (
    respicast_pred.sort_values(
        [
            "model", "disease", "location", "origin", "target", "h",
            "median_priority", "file_date"
        ],
        ascending=[True, True, True, True, True, True, True, False]
    )
    .drop_duplicates(["model", "disease", "location", "origin", "target", "h"], keep="first")
    .reset_index(drop=True)
)

print("\nRespiCast usable predictions after corrected mapping:", respicast_pred.shape)
print("RespiCast models:", respicast_pred["model"].nunique())

print("\nRespiCast h values after corrected mapping:")
display(
    respicast_pred.groupby(["model", "h"], as_index=False)
    .agg(n_rows=("pred", "size"))
    .sort_values(["model", "h"])
)

print("\nTop RespiCast models by usable rows:")
display(
    respicast_pred.groupby("model", as_index=False)
    .agg(
        rows=("pred", "size"),
        ari_rows=("disease", lambda x: int((x == "ARI").sum())),
        ili_rows=("disease", lambda x: int((x == "ILI").sum())),
        n_horizons=("h", "nunique"),
        h_values=("h", lambda x: ",".join(map(str, sorted(x.unique())))),
        first_origin=("origin", "min"),
        last_origin=("origin", "max"),
        first_target=("target", "min"),
        last_target=("target", "max")
    )
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)


# ============================================================
# 6. Align RespiCast to TFG truth universe
# ============================================================

respicast_aligned = respicast_pred.merge(
    truth_universe,
    on=truth_keys,
    how="inner"
)

respicast_aligned = respicast_aligned[
    truth_keys + [
        "y", "pred", "model", "source", "method",
        "median_source_type", "file_name",
        "origin_raw", "target_raw", "respicast_horizon_raw", "respicast_horizon_num"
    ]
].copy()

print("\n============================================================")
print("RespiCast aligned to cleaned TFG truth universe")
print("============================================================")
print("Aligned rows:", respicast_aligned.shape)

print("\nAligned RespiCast coverage by model:")
display(
    respicast_aligned.groupby(["model"], as_index=False)
    .agg(
        n_points=("pred", "size"),
        n_countries=("location", "nunique"),
        n_diseases=("disease", "nunique"),
        h_values=("h", lambda x: ",".join(map(str, sorted(x.unique())))),
        first_origin=("origin", "min"),
        last_origin=("origin", "max"),
        first_target=("target", "min"),
        last_target=("target", "max")
    )
    .sort_values("n_points", ascending=False)
    .reset_index(drop=True)
)

print("\nAligned RespiCast coverage by disease/h:")
display(
    respicast_aligned.groupby(["model", "disease", "h"], as_index=False)
    .agg(n_points=("pred", "size"), n_countries=("location", "nunique"))
    .sort_values(["disease", "h", "n_points"], ascending=[True, True, False])
)


# ============================================================
# 7. Combine TFG and RespiCast predictions
# ============================================================

tfg_combined = tfg_predictions.copy()
tfg_combined["median_source_type"] = "TFG_point"
tfg_combined["file_name"] = ""
tfg_combined["origin_raw"] = pd.NaT
tfg_combined["target_raw"] = pd.NaT
tfg_combined["respicast_horizon_raw"] = ""
tfg_combined["respicast_horizon_num"] = np.nan

combined = pd.concat(
    [
        tfg_combined,
        respicast_aligned
    ],
    ignore_index=True
)

combined = combined.merge(
    mase_scales[["disease", "location", "mase_scale"]],
    on=["disease", "location"],
    how="left"
)

missing_scale = combined[combined["mase_scale"].isna()][["disease", "location"]].drop_duplicates()

if len(missing_scale) > 0:
    print("\nMissing MASE scales:")
    display(missing_scale)
    raise ValueError("Some combined rows do not have MASE scale.")

combined["error"] = combined["y"] - combined["pred"]
combined["abs_error"] = combined["error"].abs()
combined["squared_error"] = combined["error"] ** 2
combined["scaled_abs_error"] = combined["abs_error"] / combined["mase_scale"]

combined = combined[
    np.isfinite(combined["pred"]) &
    np.isfinite(combined["y"]) &
    np.isfinite(combined["abs_error"]) &
    np.isfinite(combined["scaled_abs_error"])
].copy()

print("\nCombined table:", combined.shape)
print("TFG methods:", combined[combined["source"] == "TFG"]["method"].nunique())
print("RespiCast methods:", combined[combined["source"] == "RespiCast"]["method"].nunique())


# ============================================================
# 8. Select reference model
# ============================================================

available_respicast_methods = (
    combined[combined["source"] == "RespiCast"][["model", "method"]]
    .drop_duplicates()
    .sort_values("model")
)

ref_candidates = available_respicast_methods[
    available_respicast_methods["model"].str.lower().str.contains(
        REFERENCE_MODEL_CONTAINS.lower(),
        regex=False
    )
].copy()

if len(ref_candidates) == 0:
    print("\nAvailable RespiCast methods:")
    display(available_respicast_methods)
    raise ValueError("Could not find respicast-quantileBaseline reference method.")

REFERENCE_MODEL = ref_candidates.iloc[0]["model"]
REFERENCE_METHOD = ref_candidates.iloc[0]["method"]

print("\n============================================================")
print("Reference model")
print("============================================================")
print("REFERENCE_MODEL:", REFERENCE_MODEL)
print("REFERENCE_METHOD:", REFERENCE_METHOD)


# ============================================================
# 9. Build exact paired log2 skill rows
# ============================================================

def build_skill_rows_for_scope(df_scope, scope_name):
    """
    Build exact pairs between each method and the reference baseline.
    The reference defines the comparison universe.
    """
    ref = df_scope[df_scope["method"] == REFERENCE_METHOD].copy()

    if len(ref) == 0:
        return pd.DataFrame()

    ref = ref[truth_keys + ["abs_error", "scaled_abs_error", "pred"]].rename(
        columns={
            "abs_error": "abs_error_ref",
            "scaled_abs_error": "scaled_abs_error_ref",
            "pred": "pred_ref"
        }
    )

    rows = []

    for method, sub in df_scope.groupby("method"):
        sub = sub.copy()

        paired = sub.merge(ref, on=truth_keys, how="inner")

        if len(paired) == 0:
            continue

        paired["scope"] = scope_name

        paired["log2_skill_vs_ref"] = safe_log2_ratio(
            paired["abs_error_ref"].values,
            paired["abs_error"].values,
            eps=EPS_AE
        )

        nonzero_mask = (paired["abs_error_ref"] > 0) & (paired["abs_error"] > 0)

        paired["log2_skill_vs_ref_nonzero_only"] = np.nan
        paired.loc[nonzero_mask, "log2_skill_vs_ref_nonzero_only"] = np.log2(
            paired.loc[nonzero_mask, "abs_error_ref"] /
            paired.loc[nonzero_mask, "abs_error"]
        )

        rows.append(paired)

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


combined_global = combined.copy()
combined_common6 = combined[combined["location"].isin(COMMON6)].copy()

skill_rows_global = build_skill_rows_for_scope(combined_global, "global")
skill_rows_common6 = build_skill_rows_for_scope(combined_common6, "common6")

skill_rows = pd.concat(
    [skill_rows_global, skill_rows_common6],
    ignore_index=True
)

if len(skill_rows) == 0:
    raise ValueError("No paired skill rows generated.")

print("\nSkill rows:", skill_rows.shape)

print("\nReference universe by scope/disease/h:")
reference_universe_counts = (
    skill_rows[skill_rows["method"] == REFERENCE_METHOD]
    .groupby(["scope", "disease", "h"], as_index=False)
    .agg(
        reference_points=("pred", "size"),
        reference_countries=("location", "nunique"),
        first_origin=("origin", "min"),
        last_origin=("origin", "max"),
        first_target=("target", "min"),
        last_target=("target", "max")
    )
)
display(reference_universe_counts.sort_values(["scope", "disease", "h"]))


# ============================================================
# 10. Metric summaries on paired reference universe
# ============================================================

def build_reference_universe_counts(skill_df):
    ref = skill_df[skill_df["method"] == REFERENCE_METHOD].copy()

    ref_counts = (
        ref.groupby(["scope", "disease", "h"], as_index=False)
        .agg(
            reference_points=("pred", "size"),
            reference_countries=("location", "nunique")
        )
    )

    return ref_counts


def build_tfg_full_universe_counts(scope_name):
    if scope_name == "global":
        base = truth_universe.copy()
    elif scope_name == "common6":
        base = truth_universe[truth_universe["location"].isin(COMMON6)].copy()
    else:
        raise ValueError("scope must be global or common6")

    out = (
        base.groupby(["disease", "h"], as_index=False)
        .agg(
            tfg_full_test_points=("y", "size"),
            tfg_full_test_countries=("location", "nunique")
        )
    )
    out["scope"] = scope_name
    return out


def summarize_skill_by_disease_h(skill_df):
    ref_counts = build_reference_universe_counts(skill_df)

    tfg_counts = pd.concat(
        [
            build_tfg_full_universe_counts("global"),
            build_tfg_full_universe_counts("common6")
        ],
        ignore_index=True
    )

    panel = (
        skill_df.groupby(["scope", "source", "model", "method", "disease", "h"], as_index=False)
        .agg(
            mean_log2_skill_panel=("log2_skill_vs_ref", "mean"),
            median_log2_skill_panel=("log2_skill_vs_ref", "median"),
            mean_log2_skill_nonzero_only=("log2_skill_vs_ref_nonzero_only", "mean"),
            n_nonzero_pairs=("log2_skill_vs_ref_nonzero_only", lambda x: int(x.notna().sum())),
            mean_AE_model=("abs_error", "mean"),
            mean_AE_ref=("abs_error_ref", "mean"),
            median_AE_model=("abs_error", "median"),
            median_AE_ref=("abs_error_ref", "median"),
            mean_scaled_AE_model=("scaled_abs_error", "mean"),
            mean_scaled_AE_ref=("scaled_abs_error_ref", "mean"),
            RMSE_model=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
            n_pair=("pred", "size"),
            n_countries=("location", "nunique"),
            first_origin=("origin", "min"),
            last_origin=("origin", "max"),
            first_target=("target", "min"),
            last_target=("target", "max")
        )
    )

    country_level = (
        skill_df.groupby(
            ["scope", "source", "model", "method", "disease", "h", "location"],
            as_index=False
        )
        .agg(country_mean_log2_skill=("log2_skill_vs_ref", "mean"))
    )

    country_balanced = (
        country_level.groupby(["scope", "source", "model", "method", "disease", "h"], as_index=False)
        .agg(mean_log2_skill_country_balanced=("country_mean_log2_skill", "mean"))
    )

    out = panel.merge(
        country_balanced,
        on=["scope", "source", "model", "method", "disease", "h"],
        how="left"
    )

    out = out.merge(
        ref_counts,
        on=["scope", "disease", "h"],
        how="left"
    )

    out = out.merge(
        tfg_counts,
        on=["scope", "disease", "h"],
        how="left"
    )

    out["coverage_vs_reference"] = out["n_pair"] / out["reference_points"]
    out["coverage_vs_tfg_full_test"] = out["n_pair"] / out["tfg_full_test_points"]

    out["passes_reference_coverage_threshold"] = (
        out["coverage_vs_reference"] >= MIN_COVERAGE_VS_REFERENCE
    )

    out["rank_by_panel_log2_skill"] = (
        out.groupby(["scope", "disease", "h"])["mean_log2_skill_panel"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    out["rank_by_country_balanced_log2_skill"] = (
        out.groupby(["scope", "disease", "h"])["mean_log2_skill_country_balanced"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    out = (
        out.sort_values(
            ["scope", "disease", "h", "rank_by_panel_log2_skill", "mean_log2_skill_panel"],
            ascending=[True, True, True, True, False]
        )
        .reset_index(drop=True)
    )

    return out


def summarize_overall(skill_df):
    panel = (
        skill_df.groupby(["scope", "source", "model", "method"], as_index=False)
        .agg(
            mean_log2_skill_panel=("log2_skill_vs_ref", "mean"),
            median_log2_skill_panel=("log2_skill_vs_ref", "median"),
            mean_log2_skill_nonzero_only=("log2_skill_vs_ref_nonzero_only", "mean"),
            n_nonzero_pairs=("log2_skill_vs_ref_nonzero_only", lambda x: int(x.notna().sum())),
            mean_AE_model=("abs_error", "mean"),
            mean_AE_ref=("abs_error_ref", "mean"),
            mean_scaled_AE_model=("scaled_abs_error", "mean"),
            mean_scaled_AE_ref=("scaled_abs_error_ref", "mean"),
            n_pair=("pred", "size"),
            n_diseases=("disease", "nunique"),
            n_horizons=("h", "nunique"),
            h_values=("h", lambda x: ",".join(map(str, sorted(x.unique())))),
            n_countries=("location", "nunique"),
            first_origin=("origin", "min"),
            last_origin=("origin", "max")
        )
    )

    country_level = (
        skill_df.groupby(
            ["scope", "source", "model", "method", "disease", "h", "location"],
            as_index=False
        )
        .agg(country_mean_log2_skill=("log2_skill_vs_ref", "mean"))
    )

    country_balanced = (
        country_level.groupby(["scope", "source", "model", "method"], as_index=False)
        .agg(mean_log2_skill_country_balanced=("country_mean_log2_skill", "mean"))
    )

    out = panel.merge(
        country_balanced,
        on=["scope", "source", "model", "method"],
        how="left"
    )

    out["rank_by_panel_log2_skill"] = (
        out.groupby("scope")["mean_log2_skill_panel"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    out["rank_by_country_balanced_log2_skill"] = (
        out.groupby("scope")["mean_log2_skill_country_balanced"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    out = (
        out.sort_values(["scope", "rank_by_panel_log2_skill"])
        .reset_index(drop=True)
    )

    return out


skill_by_disease_h = summarize_skill_by_disease_h(skill_rows)
skill_overall = summarize_overall(skill_rows)

tfg_positions_by_h = (
    skill_by_disease_h[skill_by_disease_h["source"] == "TFG"]
    .sort_values(["scope", "disease", "h", "rank_by_panel_log2_skill"])
    .reset_index(drop=True)
)

tfg_positions_overall = (
    skill_overall[skill_overall["source"] == "TFG"]
    .sort_values(["scope", "rank_by_panel_log2_skill"])
    .reset_index(drop=True)
)

top10_by_disease_h = (
    skill_by_disease_h
    .sort_values(["scope", "disease", "h", "rank_by_panel_log2_skill"])
    .groupby(["scope", "disease", "h"], group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

top10_by_disease_h_coverage_filtered = (
    skill_by_disease_h[skill_by_disease_h["passes_reference_coverage_threshold"]]
    .sort_values(["scope", "disease", "h", "rank_by_panel_log2_skill"])
    .groupby(["scope", "disease", "h"], group_keys=False)
    .head(10)
    .reset_index(drop=True)
)


# ============================================================
# 11. Coverage, zero-error and extreme prediction audits
# ============================================================

coverage_all = (
    skill_by_disease_h[
        [
            "scope", "source", "model", "method", "disease", "h",
            "n_pair", "n_countries", "reference_points", "reference_countries",
            "tfg_full_test_points", "tfg_full_test_countries",
            "coverage_vs_reference", "coverage_vs_tfg_full_test",
            "first_origin", "last_origin", "first_target", "last_target"
        ]
    ]
    .sort_values(["scope", "disease", "h", "coverage_vs_reference"], ascending=[True, True, True, False])
    .reset_index(drop=True)
)

zero_ae_audit = (
    skill_rows
    .assign(
        model_ae_zero=lambda d: d["abs_error"].eq(0),
        ref_ae_zero=lambda d: d["abs_error_ref"].eq(0),
        both_zero=lambda d: d["abs_error"].eq(0) & d["abs_error_ref"].eq(0)
    )
    .groupby(["scope", "source", "model", "method", "disease", "h"], as_index=False)
    .agg(
        n_pair=("pred", "size"),
        n_model_ae_zero=("model_ae_zero", "sum"),
        n_ref_ae_zero=("ref_ae_zero", "sum"),
        n_both_zero=("both_zero", "sum")
    )
    .sort_values(["scope", "disease", "h", "source", "model"])
    .reset_index(drop=True)
)

extreme_prediction_audit = (
    combined.assign(
        abs_pred=lambda d: d["pred"].abs(),
        is_extreme=lambda d: d["pred"].abs() > EXTREME_PRED_ABS_THRESHOLD
    )
    .groupby(["source", "model", "method", "disease", "h"], as_index=False)
    .agg(
        n_points=("pred", "size"),
        n_extreme_pred=("is_extreme", "sum"),
        max_abs_pred=("abs_pred", "max"),
        min_pred=("pred", "min"),
        max_pred=("pred", "max")
    )
    .sort_values(["n_extreme_pred", "max_abs_pred"], ascending=[False, False])
    .reset_index(drop=True)
)

print("\n============================================================")
print("Main corrected outputs")
print("============================================================")

print("\nSkill by disease/horizon:")
display(skill_by_disease_h)

print("\nTFG positions by disease/horizon:")
display(tfg_positions_by_h)

print("\nTop 10 by disease/horizon, coverage filtered:")
display(top10_by_disease_h_coverage_filtered)

print("\nTFG positions overall:")
display(tfg_positions_overall)

print("\nZero AE audit with any zero cases:")
display(
    zero_ae_audit[
        (zero_ae_audit["n_model_ae_zero"] > 0) |
        (zero_ae_audit["n_ref_ae_zero"] > 0) |
        (zero_ae_audit["n_both_zero"] > 0)
    ]
)

print("\nExtreme prediction audit with any extreme predictions:")
display(extreme_prediction_audit[extreme_prediction_audit["n_extreme_pred"] > 0])


# ============================================================
# 12. Save all outputs
# ============================================================

respicast_inventory.to_csv(OUT_DIR / "01_respicast_file_inventory_corrected.csv", index=False)
respicast_pred.to_csv(OUT_DIR / "02_respicast_predictions_corrected_origin_long.csv", index=False)
respicast_aligned.to_csv(OUT_DIR / "03_respicast_predictions_aligned_to_tfg_truth_corrected.csv", index=False)

tfg_coverage.to_csv(OUT_DIR / "04_tfg_coverage_after_cleaning.csv", index=False)
truth_universe.to_csv(OUT_DIR / "05_tfg_truth_universe_after_cleaning.csv", index=False)

combined.to_csv(OUT_DIR / "06_combined_tfg_respicast_predictions_long_corrected.csv", index=False)

skill_rows.to_csv(OUT_DIR / "07_log2_skill_rows_vs_respicast_quantileBaseline_corrected.csv", index=False)
skill_by_disease_h.to_csv(OUT_DIR / "08_log2_skill_by_disease_horizon_corrected.csv", index=False)
skill_overall.to_csv(OUT_DIR / "09_log2_skill_overall_corrected.csv", index=False)

tfg_positions_by_h.to_csv(OUT_DIR / "10_tfg_positions_by_disease_horizon_corrected.csv", index=False)
tfg_positions_overall.to_csv(OUT_DIR / "11_tfg_positions_overall_corrected.csv", index=False)

top10_by_disease_h.to_csv(OUT_DIR / "12_top10_by_disease_horizon_corrected.csv", index=False)
top10_by_disease_h_coverage_filtered.to_csv(
    OUT_DIR / "13_top10_by_disease_horizon_coverage_filtered_corrected.csv",
    index=False
)

coverage_all.to_csv(OUT_DIR / "14_coverage_vs_reference_and_tfg_universe_corrected.csv", index=False)
zero_ae_audit.to_csv(OUT_DIR / "15_zero_ae_audit_corrected.csv", index=False)
extreme_prediction_audit.to_csv(OUT_DIR / "16_extreme_prediction_audit_corrected.csv", index=False)

reference_universe_counts.to_csv(OUT_DIR / "17_reference_universe_by_scope_disease_horizon.csv", index=False)

methodology_notes = f"""
Corrected TFG vs RespiCast comparison methodology

Inputs
------
TFG predictions:
{TFG_PRED_PATH}

TFG MASE scales:
{MASE_SCALE_PATH}

RespiCast model-output folder:
{MODEL_OUTPUT_DIR}

Main corrections
----------------
1. TFG predictions were cleaned by removing rows with missing observed y,
   missing prediction, or non-finite y/pred. This is necessary because some
   base model prediction tables contained forecasts for targets where the
   observed value was not available. Those rows caused coverage_rate > 1
   in the previous exploratory comparison.

2. RespiCast origin dates are publication dates, usually Wednesdays.
   The comparable TFG origin is the previous Sunday, not the next Sunday.
   Therefore:
       origin_tfg = previous Sunday at or before origin_raw
       target     = target_end_date normalized to Sunday
       h_tfg      = (target - origin_tfg) / 7

3. RespiCast horizon 1 usually maps to TFG h=0 and is excluded from the
   h=1..4 TFG comparison. RespiCast horizons 2,3,4 usually map to TFG
   h=1,2,3. Consequently, h=4 may not be available in this comparison
   unless a RespiCast file provides a longer target date.

Forecast type
-------------
RespiCast point forecasts are obtained using:
- output_type == median, or
- output_type == point / mean, or
- output_type == quantile and output_type_id == 0.5.

Reference model
---------------
{REFERENCE_METHOD}

Main score
----------
For each exactly matched forecast row:
    log2_skill = log2(AE_reference / AE_model)

Positive values indicate improvement over the RespiCast reference.
Negative values indicate worse performance than the RespiCast reference.

Pairing
-------
All log2 scores are computed only on exact matched rows:
    origin, target, disease, location, h

The reference model defines the comparison universe in each scope,
disease and horizon.

Reported coverage
-----------------
coverage_vs_reference:
    n matched rows for the method divided by the number of reference rows
    in the same scope, disease and horizon.

coverage_vs_tfg_full_test:
    n matched rows for the method divided by the full cleaned TFG test
    universe in the same scope, disease and horizon.

Zero AE handling
----------------
If AE_reference = 0 and AE_model = 0, the log2 score is set to 0.
Otherwise, epsilon = {EPS_AE} is used to avoid undefined log ratios.
A zero-error audit is saved separately.

Sensitivity
-----------
The table also reports mean_log2_skill_nonzero_only, computed only on
pairs where both AE_reference and AE_model are strictly positive.

Main output files
-----------------
08_log2_skill_by_disease_horizon_corrected.csv:
    Full ranking by scope, disease and horizon.

10_tfg_positions_by_disease_horizon_corrected.csv:
    Only TFG methods, showing their position against RespiCast.

13_top10_by_disease_horizon_coverage_filtered_corrected.csv:
    Top methods after requiring coverage_vs_reference >= {MIN_COVERAGE_VS_REFERENCE}.

14_coverage_vs_reference_and_tfg_universe_corrected.csv:
    Coverage diagnostics.

15_zero_ae_audit_corrected.csv:
    Exact zero absolute error diagnostics.

16_extreme_prediction_audit_corrected.csv:
    Extreme prediction diagnostics.
"""

with open(OUT_DIR / "00_methodology_notes_corrected.txt", "w", encoding="utf-8") as f:
    f.write(methodology_notes)

print("\n============================================================")
print("Saved corrected RespiCast comparison outputs")
print("============================================================")
print(OUT_DIR)

for p in sorted(OUT_DIR.glob("*")):
    print("-", p.name)

print("\nDone.")

PROJECT_ROOT: C:\Users\aolas\UNI PYTHON\TFG
FINAL_TEST_DIR: C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025
RESPICAST_REPO: C:\Users\aolas\UNI PYTHON\TFG\RespiCast-SyndromicIndicators-main
MODEL_OUTPUT_DIR: C:\Users\aolas\UNI PYTHON\TFG\RespiCast-SyndromicIndicators-main\model-output
OUT_DIR: C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\respicast_comparison_corrected

TFG prediction cleaning
Original rows: 49892
Rows after dropping missing y/pred and non-finite values: 49252
Rows removed: 640

TFG coverage after cleaning:


,model,disease,h,n_points,n_countries,tfg_universe_points,tfg_universe_countries,coverage_vs_tfg_universe
0,DL_GlobalGRU_all_infections_all_countries,ARI,1,791,8,791,8,1.0
8,RandomForest(lags=4),ARI,1,791,8,791,8,1.0
16,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",ARI,1,791,8,791,8,1.0
24,autoARIMA,ARI,1,791,8,791,8,1.0
32,ensemble_mean_5,ARI,1,791,8,791,8,1.0
40,ensemble_median_5,ARI,1,791,8,791,8,1.0
48,rf_global_all_infections_all_countries,ARI,1,791,8,791,8,1.0
1,DL_GlobalGRU_all_infections_all_countries,ARI,2,783,8,783,8,1.0
9,RandomForest(lags=4),ARI,2,783,8,783,8,1.0
17,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",ARI,2,783,8,783,8,1.0



TFG truth universe after cleaning:


,disease,h,n_points,n_countries,first_origin,last_origin,first_target,last_target
0,ARI,1,791,8,2024-01-07,2025-12-07,2024-01-14,2025-12-14
1,ARI,2,783,8,2024-01-07,2025-11-30,2024-01-21,2025-12-14
2,ARI,3,775,8,2024-01-07,2025-11-23,2024-01-28,2025-12-14
3,ARI,4,767,8,2024-01-07,2025-11-16,2024-02-04,2025-12-14
4,ILI,1,995,10,2024-01-07,2025-12-07,2024-01-14,2025-12-14
5,ILI,2,985,10,2024-01-07,2025-11-30,2024-01-21,2025-12-14
6,ILI,3,975,10,2024-01-07,2025-11-23,2024-01-28,2025-12-14
7,ILI,4,965,10,2024-01-07,2025-11-16,2024-02-04,2025-12-14



MASE scales loaded: (18, 3)

Loading RespiCast forecasts
CSV files found: 1240
Processed 200/1240 files...
Processed 400/1240 files...
Processed 600/1240 files...
Processed 800/1240 files...
Processed 1000/1240 files...
Processed 1200/1240 files...

RespiCast usable predictions after corrected mapping: (134972, 25)
RespiCast models: 28

RespiCast h values after corrected mapping:


...


Zero AE audit with any zero cases:


,scope,source,model,method,disease,h,n_pair,n_model_ae_zero,n_ref_ae_zero,n_both_zero
100,common6,RespiCast,ECDC-SARIMA,RespiCast::ECDC-SARIMA,ILI,1,290,4,15,4
101,common6,RespiCast,ECDC-soca_simplex,RespiCast::ECDC-soca_simplex,ILI,1,323,0,12,0
102,common6,RespiCast,ISI-FluABCaster,RespiCast::ISI-FluABCaster,ILI,1,159,0,1,0
103,common6,RespiCast,ISI-FluBcast,RespiCast::ISI-FluBcast,ILI,1,254,1,4,0
106,common6,RespiCast,ISI-RC_AdaptEns2,RespiCast::ISI-RC_AdaptEns2,ILI,1,99,0,3,0
...,...,...,...,...,...,...,...,...,...,...
451,global,TFG,"SARIMA(1,0,0)x(1,0,0)[52] (initial)","TFG::SARIMA(1,0,0)x(1,0,0)[52] (initial)",ILI,4,572,0,11,0
452,global,TFG,autoARIMA,TFG::autoARIMA,ILI,4,572,12,11,7
453,global,TFG,ensemble_mean_5,TFG::ensemble_mean_5,ILI,4,572,0,11,0
454,global,TFG,ensemble_median_5,TFG::ensemble_median_5,ILI,4,572,0,11,0



Extreme prediction audit with any extreme predictions:


,source,model,method,disease,h,n_points,n_extreme_pred,max_abs_pred,min_pred,max_pred
0,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,4,473,1,2.322790e+30,0.0,2.322790e+30
1,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,3,483,1,1.153267e+23,0.0,1.153267e+23
2,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,2,492,1,5.725977e+15,0.0,5.725977e+15
3,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,1,493,1,2.842952e+08,0.0,2.842952e+08



Saved corrected RespiCast comparison outputs
C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\respicast_comparison_corrected
- .ipynb_checkpoints
- 00_methodology_notes_corrected.txt
- 01_respicast_file_inventory_corrected.csv
- 02_respicast_predictions_corrected_origin_long.csv
- 03_respicast_predictions_aligned_to_tfg_truth_corrected.csv
- 04_tfg_coverage_after_cleaning.csv
- 05_tfg_truth_universe_after_cleaning.csv
- 06_combined_tfg_respicast_predictions_long_corrected.csv
- 07_log2_skill_rows_vs_respicast_quantileBaseline_corrected.csv
- 08_log2_skill_by_disease_horizon_corrected.csv
- 09_log2_skill_overall_corrected.csv
- 10_tfg_positions_by_disease_horizon_corrected.csv
- 11_tfg_positions_overall_corrected.csv
- 12_top10_by_disease_horizon_corrected.csv
- 13_top10_by_disease_horizon_coverage_filtered_corrected.csv
- 14_coverage_vs_reference_and_tfg_universe_corrected.csv
- 15_zero_ae_audit_corrected.csv
- 16_extreme_prediction_audit_corrected.csv
- 17_reference_univers